# Bronze — PostgreSQL mappings and statement events

Read a demo mapping snapshot, consume the matching Redpanda event, and preserve the source bytes in MinIO. Run after seeding.

In [1]:
import hashlib
import json
import os
import sys
from decimal import Decimal
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4

import boto3
import requests
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import dotenv_values
from pyspark.sql import SparkSession, functions as F

# Support launching Jupyter from either the repository root or the notebook folder.
cwd = Path.cwd().resolve()
candidates = [parent if parent.name == "data-platform" else parent / "data-platform"
              for parent in (cwd, *cwd.parents)]
env_path = next((folder / ".env" for folder in candidates if (folder / ".env").is_file()), None)
if env_path is None:
    raise FileNotFoundError("Create data-platform/.env and launch from the repository.")
cfg = dotenv_values(env_path, interpolate=False)

def required(name):
    value = cfg.get(name)
    if not value:
        raise ValueError(f"Set {name} in data-platform/.env")
    return value

endpoint = cfg.get("MINIO_ENDPOINT") or "http://192.168.29.250:9000"
bucket = cfg.get("BRONZE_BUCKET") or "bronze"
os.environ["AWS_ACCESS_KEY_ID"] = required("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = required("AWS_SECRET_ACCESS_KEY")
os.environ["PYSPARK_PYTHON"] = sys.executable

# Use path-style addressing for both Spark and the raw-object client.
s3 = boto3.client(
    "s3", endpoint_url=endpoint, region_name="us-east-1",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    config=Config(s3={"addressing_style": "path"}),
)
spark = (
    SparkSession.builder.master("local[2]").appName("bronze-erpnext-gl")
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.3.3,org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.hadoop.fs.s3a.endpoint", endpoint)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(urlsplit(endpoint).scheme == "https").lower())
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)

26/11/16 06:30:05 WARN Utils: Your hostname, DESKTOP-8SSCKF6 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/16 06:30:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/kirit/.venvs/bronze-py311/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/kirit/.ivy2/cache
The jars for the packages stored in: /home/kirit/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d478c10d-aaef-4093-b76d-d5064c026a8a;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 3073ms :: artifacts dl 800ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.3.3 from central in [default]
	io.delta#delta-storage;3.3.3 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hado

In [2]:
import time
import psycopg
from psycopg.rows import dict_row
from confluent_kafka import Consumer

run = required("DEMO_RUN_ID")
with psycopg.connect(required("PG_DSN"), row_factory=dict_row) as db:
    mappings = db.execute("""SELECT scenario, company, ledger_account, bank_account_id
        FROM silver_demo_account_mapping WHERE scenario_run = %s
        ORDER BY scenario, bank_account_id""", (run,)).fetchall()
if not mappings:
    raise RuntimeError("Seed PostgreSQL first.")

# Use a temporary group for this bounded demo and leave shared offsets alone.
consumer = Consumer({"bootstrap.servers": required("REDPANDA_BOOTSTRAP_SERVERS"),
    "group.id": f"silver-demo-{uuid4().hex}", "auto.offset.reset": "earliest",
    "enable.auto.commit": False})
consumer.subscribe([required("STATEMENT_TOPIC")])
try:
    deadline = time.monotonic() + 45
    event_bytes = None
    while time.monotonic() < deadline:
        message = consumer.poll(1)
        if message is None:
            continue
        if message.error():
            raise RuntimeError(str(message.error()))
        if message.key() == f"statement-{run}".encode():
            event_bytes = message.value()
            break
finally:
    consumer.close()
if event_bytes is None:
    raise RuntimeError("No matching demo event. Check that the seed published it.")
event = json.loads(event_bytes)
expected = {"event_id": f"statement-{run}", "bucket": required("STATEMENT_BUCKET"),
            "object_key": f"silver-demo/{run}.csv"}
if any(event.get(key) != value for key, value in expected.items()):
    raise ValueError("Statement event does not match this demo run.")
with s3.get_object(Bucket=event["bucket"], Key=event["object_key"])["Body"] as body:
    csv_bytes = body.read()
for name, payload in {"mappings.json": json.dumps(mappings).encode(),
                      "event.json": event_bytes, "statement.csv": csv_bytes}.items():
    # Replaying the same content reuses the same object version by key.
    digest = hashlib.sha256(payload).hexdigest()
    key = f"raw/silver-demo/{run}/{digest}/{name}"
    try:
        s3.put_object(Bucket=bucket, Key=key, Body=payload, IfNoneMatch="*")
    except ClientError as exc:
        if exc.response["ResponseMetadata"]["HTTPStatusCode"] != 412:
            raise
    expected[name] = key
s3.put_object(Bucket=bucket, Key=f"raw/silver-demo/{run}/manifest.json",
              Body=json.dumps(expected).encode())
print("PASS: mappings, statement and event landed in Bronze")

PASS: mappings, statement and event landed in Bronze
